In [ ]:
%load_ext autoreload

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from collections import defaultdict
import itertools
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

In [ ]:
%autoreload 2
from src.data import get_electrode_df, add_metadata_features
from src.data_cleaning import prepare_ABC_results, compute_stimulus_correlation
from src.models.decoding import run_decoding_population, run_decoding_model_comparison_population

In [ ]:
sns.set(context="paper", font_scale=2)

In [ ]:
epochs_path = "outputs/epochs_preprocessed/EC260_epo.fif"

electrodes_paths = "outputs/causal4/find_speech_responsive/EC260_results.csv"

A_result_path = Path("outputs/causal4/unify_As/results.csv")

all_A_result_path = Path("outputs/causal4/find_As/EC260_results.csv")
all_A_decoders_path = Path("outputs/causal4/find_As/EC260_decoders.pt")

B_annotated_path = Path("outputs/causal4/annotated_B_results.csv")
C_annotated_path = Path("outputs/causal4/annotated_C_results.csv")

outdir = "."

A_veridical_threshold = 0.3

In [ ]:
subject = re.findall(r"(EC[\d]+)_epo", str(epochs_path))[0]

In [ ]:
electrode_df = pd.read_csv(electrodes_paths)

In [ ]:
epochs = mne.read_epochs(epochs_path, verbose=False)
assert epochs.metadata is not None
epochs.metadata = add_metadata_features(epochs.metadata)

In [ ]:
unified_A_results, B_results, C_results = prepare_ABC_results(A_result_path, B_annotated_path, C_annotated_path)

In [ ]:
B_results = B_results[B_results["subject"] == subject]

In [ ]:
C_results = C_results[C_results["subject"] == subject]

In [ ]:
A_decoders = torch.load(all_A_decoders_path)

In [ ]:
A_results = pd.read_csv(all_A_result_path).query("A and subject == @subject")

In [ ]:
A_results["stimulus_correlation"], A_outcomes = compute_stimulus_correlation(
    A_results,
    {subject: A_decoders},
    {subject: epochs},
    return_outcomes=True)

In [ ]:
A_results["veridical"] = A_results["stimulus_correlation"] > A_veridical_threshold

## Decode from B super-populations

In [ ]:
B_decoding_results, B_decoders = {}, {}
for (subject, phoneme_pair), rows in tqdm(B_results.groupby(["subject", "phoneme_pair"])):
    elec_idxs = rows.electrode_idx
    assert elec_idxs.nunique() == len(elec_idxs)

    key = (subject, phoneme_pair)
    B_decoding_results[key], B_decoders[key] = run_decoding_model_comparison_population(
        epochs,
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name="B_super",
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components="auto",
        strategy="train-test",
        groupby=["word_end"],
        return_estimators=True,
    )

## Decode from C super-populations

In [ ]:
C_decoding_results, C_decoders = {}, {}
for (subject, phoneme_pair), rows in tqdm(C_results.groupby(["subject", "phoneme_pair"])):
    elec_idxs = rows.electrode_idx.unique().tolist()

    key = (subject, phoneme_pair)
    C_decoding_results[key], C_decoders[key] = run_decoding_model_comparison_population(
        epochs,
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name="C_super",
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components="auto",
        strategy="train-test",
        groupby=["word_end"],
        return_estimators=True,
    )

## Decode from BC super-population

In [ ]:
BC_decoding_results, BC_decoders = {}, {}
for (subject, phoneme_pair), rows in tqdm(B_results.groupby(["subject", "phoneme_pair"])):
    C_results_i = C_results[
        (C_results["phoneme_pair"] == phoneme_pair) & (C_results["subject"] == subject)
    ]
    rows = pd.concat([rows, C_results_i], axis=0)

    elec_idxs = rows.electrode_idx.unique().tolist()

    key = (subject, phoneme_pair)
    BC_decoding_results[key], BC_decoders[key] = run_decoding_model_comparison_population(
        epochs,
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name="BC_super",
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components="auto",
        strategy="train-test",
        groupby=["word_end"],
        return_estimators=True,
    )

## Save

In [ ]:
torch.save({"B_decoding_results": B_decoding_results,
            "C_decoding_results": C_decoding_results,
            "BC_decoding_results": BC_decoding_results,
            
            "B_decoders": B_decoders,
            "C_decoders": C_decoders,
            "BC_decoders": BC_decoders,
            },
            f"{outdir}/results.pt")